In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Data Loading and Overview

In [2]:
df = pd.read_csv('C:/Users/Mohammed.Azarudeen/Downloads/archive/Amazon Sale Report.csv')

C:\Users\Mohammed.Azarudeen\AppData\Local\Temp\ipykernel_14128\1125571423.py:1: DtypeWarning: Columns (0: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:/Users/Mohammed.Azarudeen/Downloads/archive/Amazon Sale Report.csv')


In [3]:
df.shape

(128975, 24)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  str    
 2   Date                128975 non-null  str    
 3   Status              128975 non-null  str    
 4   Fulfilment          128975 non-null  str    
 5   Sales Channel       128975 non-null  str    
 6   ship-service-level  128975 non-null  str    
 7   Style               128975 non-null  str    
 8   SKU                 128975 non-null  str    
 9   Category            128975 non-null  str    
 10  Size                128975 non-null  str    
 11  ASIN                128975 non-null  str    
 12  Courier Status      122103 non-null  str    
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  str    
 15  Amount              121180 non-null  float64


In [5]:
pd.set_option('display.max_columns', None)

In [6]:
required_column = ['Order ID', 'Date', 'Status', 'Fulfilment', 'ship-service-level', 'Category', 'Qty', 'Amount', 'ship-city', 'ship-state']
wdf =  df[required_column].copy()

In [7]:
wdf.head()

,Order ID,Date,Status,Fulfilment,ship-service-level,Category,Qty,Amount,ship-city,ship-state
0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Standard,Set,0,647.62,MUMBAI,MAHARASHTRA
1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Standard,kurta,1,406.00,BENGALURU,KARNATAKA
2,404-0687676-7273146,04-30-22,Shipped,Amazon,Expedited,kurta,1,329.00,NAVI MUMBAI,MAHARASHTRA
3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Standard,Western Dress,0,753.33,PUDUCHERRY,PUDUCHERRY
4,407-1069790-7240320,04-30-22,Shipped,Amazon,Expedited,Top,1,574.00,CHENNAI,TAMIL NADU


# Data Cleaning

In [8]:
wdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Order ID            128975 non-null  str    
 1   Date                128975 non-null  str    
 2   Status              128975 non-null  str    
 3   Fulfilment          128975 non-null  str    
 4   ship-service-level  128975 non-null  str    
 5   Category            128975 non-null  str    
 6   Qty                 128975 non-null  int64  
 7   Amount              121180 non-null  float64
 8   ship-city           128942 non-null  str    
 9   ship-state          128942 non-null  str    
dtypes: float64(1), int64(1), str(8)
memory usage: 9.8 MB


In [9]:
# change the datatypes
wdf['Date'] = pd.to_datetime(wdf['Date'], format ='%m-%d-%y')
wdf['Month_Year'] = wdf['Date'].dt.to_period('M')

# Convert Pandas Period to string for safer storage and processing
wdf['Month_Year'] =  wdf['Date'].astype(str)

In [10]:
wdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Order ID            128975 non-null  str           
 1   Date                128975 non-null  datetime64[us]
 2   Status              128975 non-null  str           
 3   Fulfilment          128975 non-null  str           
 4   ship-service-level  128975 non-null  str           
 5   Category            128975 non-null  str           
 6   Qty                 128975 non-null  int64         
 7   Amount              121180 non-null  float64       
 8   ship-city           128942 non-null  str           
 9   ship-state          128942 non-null  str           
 10  Month_Year          128975 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(8)
memory usage: 10.8 MB


In [11]:
wdf.loc[wdf['Amount'].isna(), 'Status'].value_counts()

Status
Cancelled                       7566
Shipped                          208
Shipped - Delivered to Buyer       8
Shipping                           8
Shipped - Returned to Seller       3
Pending                            2
Name: count, dtype: int64

In [12]:
wdf['Status'].value_counts()

Status
Shipped                          77804
Shipped - Delivered to Buyer     28769
Cancelled                        18332
Shipped - Returned to Seller      1953
Shipped - Picked Up                973
Pending                            658
Pending - Waiting for Pick Up      281
Shipped - Returning to Seller      145
Shipped - Out for Delivery          35
Shipped - Rejected by Buyer         11
Shipping                             8
Shipped - Lost in Transit            5
Shipped - Damaged                    1
Name: count, dtype: int64

# Revenue Definitions and Order Segmentation

To avoid revenue distortion, orders were segmented into:

    1. Revenue-generating orders (successfully fulfilled)
    2. Revenue leakage orders (cancelled, returned, lost, damaged)
    3. In-transit orders (shipping, Pending, Shipped - Out for Delivery)

This separation ensures accurate revenue measurement and enables focused leakage & transit analysis.


In [13]:
#columns segmentation
revenue_status =  [ 'Shipped',
                  'Shipped - Delivered to Buyer',
                  'Shipped - Picked Up'
                  ]

leaked_status = ['Cancelled',
                 'Shipped - Returned to Seller',
                 'Shipped - Returning to Seller',
                 'Shipped - Rejected by Buyer',
                 'Shipped - Lost in Transit',
                 'Shipped - Damaged'
                ]

transit_status = ['shipping', 
                  'Pending', 
                  'Shipped - Out for Delivery', 
                  'Pending - Waiting for Pick Up'
                 ]

In [14]:
#Normalise to avoids issues like spaces, inconsistent case, etc.
wdf['Status'] = wdf['Status'].astype(str).str.strip()

In [15]:
# create UDF

def classify_status (status):
    if status in revenue_status:
        return 'revenue'
    elif status in leaked_status:
        return 'leakage'
    elif status in transit_status:  
        return 'transit'
    else: 
        return None
        
wdf ['rev_flag'] = wdf['Status'].apply(classify_status)  

In [16]:
#check if there is any record in None Status

wdf [wdf['Status'] == None]

,Order ID,Date,Status,Fulfilment,ship-service-level,Category,Qty,Amount,ship-city,ship-state,Month_Year,rev_flag


In [17]:
# splitting the data into different DFs

df_revenue = wdf[ wdf['rev_flag'] == 'revenue'].copy()
df_leakage = wdf[ wdf['rev_flag'] == 'leakage'].copy()
df_transit = wdf[ wdf['rev_flag'] == 'transit'].copy()

In [18]:
df_revenue.head()

,Order ID,Date,Status,Fulfilment,ship-service-level,Category,Qty,Amount,ship-city,ship-state,Month_Year,rev_flag
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Standard,kurta,1,406.0,BENGALURU,KARNATAKA,2022-04-30,revenue
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Expedited,kurta,1,329.0,NAVI MUMBAI,MAHARASHTRA,2022-04-30,revenue
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Expedited,Top,1,574.0,CHENNAI,TAMIL NADU,2022-04-30,revenue
5,404-1490984-4578765,2022-04-30,Shipped,Amazon,Expedited,Set,1,824.0,GHAZIABAD,UTTAR PRADESH,2022-04-30,revenue
6,408-5748499-6859555,2022-04-30,Shipped,Amazon,Expedited,Set,1,653.0,CHANDIGARH,CHANDIGARH,2022-04-30,revenue


In [19]:
df_leakage.head()

,Order ID,Date,Status,Fulfilment,ship-service-level,Category,Qty,Amount,ship-city,ship-state,Month_Year,rev_flag
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Standard,Set,0,647.62,MUMBAI,MAHARASHTRA,2022-04-30,leakage
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Standard,Western Dress,0,753.33,PUDUCHERRY,PUDUCHERRY,2022-04-30,leakage
8,407-5443024-5233168,2022-04-30,Cancelled,Amazon,Expedited,Set,0,NaN,HYDERABAD,TELANGANA,2022-04-30,leakage
23,404-6019946-2909948,2022-04-30,Cancelled,Merchant,Standard,Set,0,570.48,pune,MAHARASHTRA,2022-04-30,leakage
29,404-5933402-8801952,2022-04-30,Cancelled,Merchant,Standard,kurta,0,NaN,GUWAHATI,ASSAM,2022-04-30,leakage


In [20]:
df_transit.head()

,Order ID,Date,Status,Fulfilment,ship-service-level,Category,Qty,Amount,ship-city,ship-state,Month_Year,rev_flag
15631,406-6849237-1026769,2022-04-21,Shipped - Out for Delivery,Merchant,Standard,Set,1,1115.0,PAKYONG,SIKKIM,2022-04-21,transit
44043,405-0700572-9554710,2022-04-04,Pending,Amazon,Expedited,kurta,1,399.0,UTTARKASHI,UTTARAKHAND,2022-04-04,transit
56068,402-6584761-5651552,2022-05-26,Pending,Amazon,Expedited,Ethnic Dress,1,999.0,PAKYONG,SIKKIM,2022-05-26,transit
58985,403-7066611-8150715,2022-05-24,Pending,Amazon,Expedited,kurta,1,315.0,PUNE,MAHARASHTRA,2022-05-24,transit
58986,403-7066611-8150715,2022-05-24,Pending,Amazon,Expedited,Set,0,NaN,PUNE,MAHARASHTRA,2022-05-24,transit


In [21]:
df_revenue['Status'].value_counts()

Status
Shipped                         77804
Shipped - Delivered to Buyer    28769
Shipped - Picked Up               973
Name: count, dtype: int64

## Revenue df check

In [22]:
df_revenue.groupby( 'Status' )['Amount'].agg(['sum', 'count'])

,sum,count
Status,,
Shipped,50324255.0,77596
Shipped - Delivered to Buyer,18650815.0,28761
Shipped - Picked Up,661252.0,973


In [23]:
df_revenue['Status'].value_counts()
#77804 - 77596

Status
Shipped                         77804
Shipped - Delivered to Buyer    28769
Shipped - Picked Up               973
Name: count, dtype: int64

In [24]:
# Shipped - 77804 => 77596 = 208
# Shipped - DOB => 28769 - 28761 = 8

df_revenue.groupby( 'Status' )['Amount'].apply(lambda x : x.isna().sum())

Status
Shipped                         208
Shipped - Delivered to Buyer      8
Shipped - Picked Up               0
Name: Amount, dtype: int64

In [25]:
len(df_revenue)

107546

## leakage df check

In [26]:
df_leakage.groupby( 'Status' )['Amount'].agg(['sum', 'count'])

,sum,count
Status,,
Cancelled,6919284.3,10766
Shipped - Damaged,1136.0,1
Shipped - Lost in Transit,1997.0,5
Shipped - Rejected by Buyer,7295.0,11
Shipped - Returned to Seller,1269644.0,1950
Shipped - Returning to Seller,107620.0,145


In [27]:
df_leakage['Status'].value_counts()

Status
Cancelled                        18332
Shipped - Returned to Seller      1953
Shipped - Returning to Seller      145
Shipped - Rejected by Buyer         11
Shipped - Lost in Transit            5
Shipped - Damaged                    1
Name: count, dtype: int64

In [28]:
#check NaN
df_leakage.groupby('Status')['Amount'].apply( lambda x : x.isna().sum())

Status
Cancelled                        7566
Shipped - Damaged                   0
Shipped - Lost in Transit           0
Shipped - Rejected by Buyer         0
Shipped - Returned to Seller        3
Shipped - Returning to Seller       0
Name: Amount, dtype: int64

## transit df check

In [29]:
df_transit.groupby( 'Status' )['Amount'].agg(['sum', 'count'])

,sum,count
Status,,
Pending,430271.0,656
Pending - Waiting for Pick Up,192138.0,281
Shipped - Out for Delivery,26971.0,35


In [30]:
df_transit['Status'].value_counts()

Status
Pending                          658
Pending - Waiting for Pick Up    281
Shipped - Out for Delivery        35
Name: count, dtype: int64

In [31]:
df_transit.groupby( 'Status' )['Amount'].apply( lambda x : x.isna().sum())

Status
Pending                          2
Pending - Waiting for Pick Up    0
Shipped - Out for Delivery       0
Name: Amount, dtype: int64

In [32]:
len(df_revenue), len(df_leakage), len(df_transit)

(107546, 20447, 974)

In [33]:
df_revenue.info()

<class 'pandas.DataFrame'>
Index: 107546 entries, 1 to 128974
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Order ID            107546 non-null  str           
 1   Date                107546 non-null  datetime64[us]
 2   Status              107546 non-null  str           
 3   Fulfilment          107546 non-null  str           
 4   ship-service-level  107546 non-null  str           
 5   Category            107546 non-null  str           
 6   Qty                 107546 non-null  int64         
 7   Amount              107330 non-null  float64       
 8   ship-city           107523 non-null  str           
 9   ship-state          107523 non-null  str           
 10  Month_Year          107546 non-null  str           
 11  rev_flag            107546 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(9)
memory usage: 10.7 MB


# Write to Potgres Engine

In [34]:

db_password = os.getenv("DB_PASSWORD")

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=db_password,
    host="localhost",
    port=5432,
    database="postgres",
)

engine = create_engine(url)

In [35]:
#created a empty table
df_revenue.head(0).to_sql('orders_revenue', con=engine, index=False, if_exists='replace')
df_leakage.head(0).to_sql('orders_leakage', con=engine, index=False, if_exists='replace')
df_transit.head(0).to_sql('orders_transit', con=engine, index=False, if_exists='replace')


0

In [36]:
#On future refreshes → TRUNCATE + append

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE orders_revenue"))
    conn.execute(text("TRUNCATE TABLE orders_leakage"))
    conn.execute(text("TRUNCATE TABLE orders_transit"))
    


In [37]:
df_revenue.to_sql(
    'orders_revenue',
    con=engine,
    index=False,
    if_exists='append'
)

df_leakage.to_sql(
    'orders_leakage',
    con=engine,
    index=False,
    if_exists='append'
)

df_transit.to_sql(
    'orders_transit',
    con=engine,
    index=False,
    if_exists='append'
)

974